# Preparing Review Bundles
We present exactly five Review Bundles, each containing exactly two different CUAD categories, for a total of 10 categories. Each bundle contains:
- exactly 5 positive source-text excerpts, ie a span that answers the question for a category
- exactly 5 negative source-text excerpts, ie a span with text that does not answer the category question, but can look similar

## Importing training split
We select the categories from only the training split to avoid data leakage.

In [114]:
import json
import pandas as pd
import matplotlib.pyplot as plt

In [115]:
# import the training dataset and convert to a dataframe

cleaned_df = pd.read_csv('../data/cuad/cleaned_train_df.csv')
with open('../data/splits/train.json') as f:
    train_split = json.load(f)

# verify number of titles in training split against 
train_titles = {contract['title'] for contract in train_split['data']}

# the original cleaned_train_df may contain validation contracts too, so eliminate those:
CLEANED_DF_CONTAINS_LEAK = cleaned_df['title'].nunique() > len(train_titles)

if CLEANED_DF_CONTAINS_LEAK:
    print('Warning: cleaned_train_df contains contracts not in the train split.')
    print('Filtering the cleaned_train_df to only include contracts in the train split.')
    cleaned_train_df = cleaned_df[cleaned_df['title'].isin(train_titles)]

    # and verify that the cleaned_train_df contains the correct # of contracts
    assert(cleaned_train_df['title'].nunique() == len(train_titles))

    # export the filtered cleaned_df, and rename the original
    cleaned_df.to_csv('../data/cuad/cleaned_df.csv', index=False)
    cleaned_train_df.to_csv('../data/cuad/cleaned_train_df.csv', index=False)



## Selecting and grouping categories
We previously selected 10 core categories on the following metrics (more in 01_EDA.ipynb), but on the unfiltered dataset:

- **Class balance**: absolute number of answered (non-impossible) examples per category. A low impossibility rate alone doesn't automatically make a category a good candidate to focus on; categories can contain few impossible q/as but be infrequent overall. Absolute number of answerable examples balances frequency and impossibility. From the September milestone doc, we add the following requirement:
    - Each category has at least 20 positive Contracts and 20 positive context groups. Exceptions must be documented and justified.
    - **Positive contract definition**: a contract containing at least one instance of an answerable question of the selected category
    - **Positive context group definition:** a group of one or more contracts, containing duplicate contracts, containing at least one instance of an answerable question of the selected category

- **Label quality/consistency**: coefficient of variation (std/mean) of `answer_text` length. Unlike variance and standard deviation, CV is a unitless measure that allows us to account for differences in mean answer/clause length and directly compare variation across categories. A high CV suggests annotators captured wildly different granularities under the same category (e.g. sometimes a single date, sometimes a whole clause), which is noisier to model and evaluate.

- **Span extraction difficulty**: median word count of `answer_text`. Very short spans (names, dates) are close to trivial extraction; very long spans are harder to extract cleanly, as they are liable to be broken across multiple chunks. Chunking for mid-length, or median clause-length, spans allows each chunk to contain clauses while not blurring multiple clauses into single units of information. Choosing categories with substantial-length, low-variation spans allows for simpler chunking strategies down the line.

- **Redundancy**: several categories cluster around the same legal concept (e.g. `Exclusivity` / `Non-Compete` / `Competitive Restriction Exception`, or `Cap On Liability` / `Uncapped Liability`, or the `License Grant` / `Non-Transferable License` / `Affiliate License` family). We prefer *two* strong representatives per cluster rather than several near-duplicates, to reduce noise and expand model versatility. Strong representatives display the above traits. We also consider semantic grouping of other important categories.

We alter the redundancy metric slightly, given that the [September milestone doc's]('../docs\coach/september-execution/Accenture-Sept_milestone.md') definition of a review bundle requires two related categories within each bundle. 

We add a few metrics from the [September milestone doc]('../docs\coach/september-execution/Accenture-Sept_milestone.md'):

Rerunning previous metric checks on the training set only:

### Analyzing metrics by category

In [116]:
# use only the train split contracts filtered above
df = cleaned_train_df.copy()

df['num_words'] = df['answer_text'].str.split().str.len()

category_stats = df.groupby('category').agg(
    n_answered=('has_answer', 'sum'),
    impossible_rate=('is_impossible', 'mean'),
    words_median=('num_words', 'median'),
    len_mean=('answer_len', 'mean'),
    len_std=('answer_len', 'std'),
).sort_values('n_answered', ascending=False)

category_stats['len_cv'] = category_stats['len_std'] / category_stats['len_mean']

category_stats.round(2)

,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv
category,,,,,,
Parties,1586,0.00,2.0,22.62,24.30,1.07
License Grant,513,0.24,52.0,423.93,294.12,0.69
Cap On Liability,432,0.25,48.0,363.50,230.50,0.63
Audit Rights,416,0.31,40.0,314.34,227.43,0.72
Anti-Assignment,404,0.18,34.5,302.33,256.15,0.85
Insurance,341,0.39,39.0,322.96,286.58,0.89
Document Name,335,0.00,3.0,29.95,19.45,0.65
Agreement Date,307,0.08,3.0,18.55,10.41,0.56
Expiration Date,302,0.17,31.0,233.44,155.90,0.67


### Analyzing Positive Contract Appearances:

In [117]:
# positive contract appearance: # of contracts where the contract contains at least one instance 
# of an answerable question of the selected category 
category_stats['positive_contract_appearances'] = df[df['has_answer']].groupby('category')['title'].nunique()
category_stats.round(2)



,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv,positive_contract_appearances
category,,,,,,,
Parties,1586,0.00,2.0,22.62,24.30,1.07,325
License Grant,513,0.24,52.0,423.93,294.12,0.69,164
Cap On Liability,432,0.25,48.0,363.50,230.50,0.63,181
Audit Rights,416,0.31,40.0,314.34,227.43,0.72,139
Anti-Assignment,404,0.18,34.5,302.33,256.15,0.85,235
Insurance,341,0.39,39.0,322.96,286.58,0.89,107
Document Name,335,0.00,3.0,29.95,19.45,0.65,326
Agreement Date,307,0.08,3.0,18.55,10.41,0.56,301
Expiration Date,302,0.17,31.0,233.44,155.90,0.67,264


### Analyzing Positive Context Group Appearances

In [118]:
# TEMP: context-group logic adapted from 01_EDA (1).ipynb ("Duplicate Context Detection" / "Category Support"),
# applied to the train split only. Replace once that branch is merged.
import hashlib

# hash each train contract's full text; identical texts share a context_group_id
train_contracts_df = pd.DataFrame(
    [{'title': c['title'], 'context': c['paragraphs'][0]['context']} for c in train_split['data']]
)
train_contracts_df['context_hash'] = train_contracts_df['context'].apply(
    lambda text: hashlib.sha256(text.encode('utf-8')).hexdigest()
)
train_contracts_df['context_group_id'] = pd.factorize(train_contracts_df['context_hash'])[0] + 1

print('Train contracts:', len(train_contracts_df))
print('Train context groups:', train_contracts_df['context_group_id'].nunique())

# positive context group appearance: # of context groups containing at least one answerable
# question of the selected category
context_group_map = train_contracts_df.set_index('title')['context_group_id']
category_stats['positive_context_group_appearances'] = (
    df[df['has_answer']].assign(context_group_id=df['title'].map(context_group_map))
    .groupby('category')['context_group_id'].nunique()
)
category_stats.round(2)

Train contracts: 326
Train context groups: 325


,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv,positive_contract_appearances,positive_context_group_appearances
category,,,,,,,,
Parties,1586,0.00,2.0,22.62,24.30,1.07,325,324
License Grant,513,0.24,52.0,423.93,294.12,0.69,164,164
Cap On Liability,432,0.25,48.0,363.50,230.50,0.63,181,181
Audit Rights,416,0.31,40.0,314.34,227.43,0.72,139,139
Anti-Assignment,404,0.18,34.5,302.33,256.15,0.85,235,235
Insurance,341,0.39,39.0,322.96,286.58,0.89,107,107
Document Name,335,0.00,3.0,29.95,19.45,0.65,326,325
Agreement Date,307,0.08,3.0,18.55,10.41,0.56,301,300
Expiration Date,302,0.17,31.0,233.44,155.90,0.67,264,263


In [119]:
# remove categories that fail the frequency/class balance checks
mask = (category_stats['n_answered'] < 100) | (category_stats['positive_contract_appearances'] < 20) | (category_stats['positive_context_group_appearances'] < 20)
print('Number of categories with < 100 answerable instances or < 20 positive contract appearances:')
print(mask.sum())

category_stats = category_stats[~mask]
print('Number of categories remaining after filtering:')
print(category_stats.shape[0])


Number of categories with < 100 answerable instances or < 20 positive contract appearances:
14
Number of categories remaining after filtering:
27


### Comparing by answer length CV:

In [120]:
# view ordering by coefficient of variation of answer length``
category_stats.sort_values('len_cv', ascending=False).round(2)
# would be nice to have this as an interactive table for easier exploration, for sorting and filtering by different metrics.

,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv,positive_contract_appearances,positive_context_group_appearances
category,,,,,,,,
Effective Date,291,0.20,4.0,58.79,90.99,1.55,255,254
Parties,1586,0.00,2.0,22.62,24.30,1.07,325,324
Non-Compete,164,0.60,43.5,392.77,377.03,0.96,77,77
Revenue/Profit Sharing,260,0.46,44.5,355.33,315.65,0.89,105,105
Insurance,341,0.39,39.0,322.96,286.58,0.89,107,107
Renewal Term,141,0.59,43.0,307.17,267.53,0.87,119,119
Anti-Assignment,404,0.18,34.5,302.33,256.15,0.85,235,235
Covenant Not To Sue,109,0.71,46.0,398.45,328.08,0.82,61,61
Warranty Duration,122,0.69,39.0,290.48,238.53,0.82,49,49


### Comparing near-duplicate categories

In [121]:
# restraint-of-trade categories
category_stats.loc[['Exclusivity', 'Non-Compete']].round(2)

,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv,positive_contract_appearances,positive_context_group_appearances
category,,,,,,,,
Exclusivity,262,0.44,44.0,338.25,221.88,0.66,116,116
Non-Compete,164,0.60,43.5,392.77,377.03,0.96,77,77


In [122]:
# liability categories
category_stats.loc[['Cap On Liability', 'Uncapped Liability']].round(2)

,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv,positive_contract_appearances,positive_context_group_appearances
category,,,,,,,,
Cap On Liability,432,0.25,48.0,363.50,230.5,0.63,181,181
Uncapped Liability,115,0.68,65.0,434.71,228.9,0.53,78,78


In [123]:
# license cateogires
category_stats.loc[['License Grant', 'Non-Transferable License']].round(2)

,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv,positive_contract_appearances,positive_context_group_appearances
category,,,,,,,,
License Grant,513,0.24,52.0,423.93,294.12,0.69,164,164
Non-Transferable License,210,0.53,50.0,428.80,330.08,0.77,90,90


**TODO: finish updating this with train-only findings**
### Reading the tables

We can apply fairly similar conclusions from the original analysis on both training and validation data to the pure training split. This indicates a robust and balanced split.

- **Class balance**: We still prioritze categories with a high number of answerable questions (positive excerpts). We can safely rule out the 14 categories that contain either less than 100 examples or less than 20 positive contract appearances. 

- **Label consistency (`len_cv`)**: 
    - `Effective Date` (~1.5) and `Source Code Escrow` (~1.4) have highly inconsistent span lengths,suggesting inconsistent annotation granularity. Data from the training split has higher annotation variation than the combined train-validation set, which is expected given a smaller number of documents.
    - `Governing Law`, `Uncapped Liability`, and `Agreement` are the most consistent (lowest CV) categories with over 100 answerable examples.
- **Span difficulty (`words_median`)**: 
    - `Parties`, `Document Name`, `Agreement Date`, `Effective Date` are trivially short (2-4 words), which are simple to extract but not very representative of a typical legal clause. 
    - `License Grant`, `Cap On Liability`, `Audit Rights`, `Post-Termination Services` ...
- **Redundancy**: among the viable candidates, avoid picking more than one from the same cluster. Overlapping categories include:
    - Trade restraint: `Exclusivity` / `Non-Compete` / `Competitive Restriction Exception`
        - `Exclusivity` has the most answerable questions, lowest impossibility rate, and lowest coefficient of variance, with `Non-Compete` as the second strongest representative.
    - Liability: `Cap On Liability` / `Uncapped Liability`
        - `Cap On Liability` has a slightly higher coefficient of variance (0.67 vs 0.55), but it has a much higher number of answerable examples and much lower impossibility rate, making it a better candidate overall.
    - Licenses: `License Grant` / `Non-Transferable License` / `Affiliate License-Licensor`
        - `License Grant` has the highest number of answerables, the lowest impossibility, and the lowest coefficient of variance. `Non-Transferable License` has the 



## Bundling Categories
Each review bundle must contain two categories. For bundling, we balance semantic similarity with statistical similarity:

**Semantic Similarity:** The categories fall under a similar topic, eg `Cap On Liability` and `Uncapped Liability` both relate to liability. Semantic similarity allows for more straightforward explanations to nontechnical stakeholders.

**Statistical Similarity:** Two categories have a high co-occurence, and score highly on some similarity metric, eg Jaccard similarity, 

### 6 Candidate Review Bundles:
In no order of importance:

#### Bundle 1: Contract Metadata
Characterized by a high number of answerable examples and near-zero impossibility rate within the training split.\
**Categories**
- Parties
- Document Name

// TODO: these date types don't actually seem that similar. consider a diff grouping
#### Bundle 2: Dates
**Categories**
- Agreement Date
- Expiration Date

#### Bundle 3: Licensing
**Categories**
- License Grant
- Non-Transferable License

#### Bundle 4: Trade Restraint
**Categories**
- Exclusivity
- Non-Compete

#### Bundle 5: Liability
**Categories**
- Cap on Liability
- Uncapped Liability

#### Bundle 6: 

## Finding Source-Text Excerpts
### Positive Source-Text Excerpts
Positive source-text excerpts are representative annotated spans that answer the selected category. Each category will be represented with exactly five such excerpts. We define 'representative' by the following metrics:

- **Length:** the annotated span's length should fall between the first and third quartiles of the category's span length distribution. We use word count as a the measure for span length, since raw character length risks truncating words.

- **Clean boundaries:** for clause categories, the span should be a full sentence: it starts with a capital letter, digit, or opening bracket/quote, and ends with `.`, `!` or `?`. Spans cut off mid-sentence make poor excerpts for a reviewer.

- **Presence of characteristic terms:** select for spans containing words specific to the clause category.

- **TF-IDF cosine similarity**: select for spans with the highest TF-IDF cosine similarity to other spans in the category. This essentially filters for central or generic spans.


### Display Length Metrics



In [124]:
TENTATIVE_CATEGORIES = ['Parties', 'Document Name', 'Agreement Date', 'Expiration Date', 'Effective Date',
                        'License Grant', 'Non-Transferable License', 'Exclusivity', 'Non-Compete',
                        'Cap On Liability', 'Uncapped Liability']

# Extract annotations from selected categories
annotations = df.loc[
    df['has_answer'] & df['category'].isin(TENTATIVE_CATEGORIES),
    ['id', 'title', 'category', 'answer_text', 'answer_start', 'answer_len', 'num_words']
]

# First quartile, median, and third quartile of span length (characters and words) per category
span_length_quartiles = annotations.groupby('category').agg(
    n_spans=('answer_text', 'size'),
    len_q1=('answer_len', lambda s: s.quantile(0.25)),
    len_median=('answer_len', 'median'),
    len_q3=('answer_len', lambda s: s.quantile(0.75)),
    words_q1=('num_words', lambda s: s.quantile(0.25)),
    words_median=('num_words', 'median'),
    words_q3=('num_words', lambda s: s.quantile(0.75)),
).reindex(TENTATIVE_CATEGORIES)

span_length_quartiles

,n_spans,len_q1,len_median,len_q3,words_q1,words_median,words_q3
category,,,,,,,
Parties,1586,8.00,16.0,28.00,1.00,2.0,4.00
Document Name,335,21.00,25.0,33.00,2.00,3.0,4.00
Agreement Date,307,13.00,16.0,22.00,3.00,3.0,5.00
Expiration Date,302,140.25,195.0,289.00,23.00,31.0,46.00
Effective Date,291,14.00,19.0,59.00,3.00,4.0,10.00
License Grant,513,240.00,352.0,529.00,36.00,52.0,82.00
Non-Transferable License,210,237.25,347.5,504.75,35.25,50.0,76.75
Exclusivity,262,184.75,289.0,426.00,29.00,44.0,63.75
Non-Compete,164,180.25,282.5,482.00,25.75,43.5,73.25


In [125]:
# mask out spans whose word count falls outside their category's interquartile range (inclusive bounds)
words_q1 = annotations['category'].map(span_length_quartiles['words_q1'])
words_q3 = annotations['category'].map(span_length_quartiles['words_q3'])
in_iqr = annotations['num_words'].between(words_q1, words_q3)

iqr_annotations = annotations[in_iqr]

# spans remaining per category after the length filter
pd.DataFrame({
    'n_spans': annotations['category'].value_counts(),
    'n_in_iqr': iqr_annotations['category'].value_counts(),
}).reindex(TENTATIVE_CATEGORIES)

,n_spans,n_in_iqr
category,,
Parties,1586,1317
Document Name,335,277
Agreement Date,307,241
Expiration Date,302,154
Effective Date,291,206
License Grant,513,265
Non-Transferable License,210,104
Exclusivity,262,132
Non-Compete,164,82


Shorter word-length categories like `Parties` and `Agreement Date` were filtered less than longer categories like `Non-Compete` or `Uncapped Liability`.

### Filter for Clean Boundaries
We want to keep spans that read as a full sentence: start with a capital letter, digit, or opening bracket/quote, and end with sentence-final punctuation (optionally followed by a closing quote/bracket).\
Metadata categories and dates (`Parties`, `Document Name`, `Agreement Date`) are names and dates rather than sentences, so they're exempt from this filter.

In [126]:
# keep spans that read as a full sentence: start with a capital letter, digit, or opening bracket/quote,
# and end with sentence-final punctuation (optionally followed by a closing quote/bracket)
FRAGMENT_CATEGORIES = ['Parties', 'Document Name', 'Agreement Date']   # names/dates, never full sentences

def has_clean_boundaries(text):
    stripped_text = text.str.strip()
    clean_start = stripped_text.str.match(r'["“(\[]*[A-Z0-9]')
    clean_end = stripped_text.str.contains(r'[.!?]["”)\]]*$')
    return clean_start & clean_end

clean_boundaries = has_clean_boundaries(iqr_annotations['answer_text']) | iqr_annotations['category'].isin(FRAGMENT_CATEGORIES)

candidate_spans = iqr_annotations[clean_boundaries].copy()

# spans remaining per category after the length and boundary filters
pd.DataFrame({
    'n_spans': annotations['category'].value_counts(),
    'n_in_iqr': iqr_annotations['category'].value_counts(),
    'n_clean_boundaries': candidate_spans['category'].value_counts(),
}).reindex(TENTATIVE_CATEGORIES)

,n_spans,n_in_iqr,n_clean_boundaries
category,,,
Parties,1586,1317,1317
Document Name,335,277,277
Agreement Date,307,241,241
Expiration Date,302,154,144
Effective Date,291,206,12
License Grant,513,265,234
Non-Transferable License,210,104,87
Exclusivity,262,132,113
Non-Compete,164,82,62


### Find Characteristic Terms
We use a weighted log-odds ratio... explain it more!!!

In [127]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# Weighted log-odds ratio with an informative Dirichlet prior (Monroe, Colaresi & Quinn 2008, "Fightin' Words").
# Each category's spans are contrasted against the spans of every other category in the train split.
# Counts are binary per span (document frequency), so a few very long spans can't dominate a category's counts.
all_spans = df.loc[df['has_answer'], ['category', 'answer_text']].reset_index(drop=True)

vectorizer = CountVectorizer(lowercase=True, binary=True, token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b', min_df=5)
span_term_matrix = vectorizer.fit_transform(all_spans['answer_text'])
vocab = vectorizer.get_feature_names_out()

total_counts = np.asarray(span_term_matrix.sum(axis=0)).ravel()
ALPHA_0 = 1000 # prior strength; larger shrinks rare terms harder toward zero
alpha = ALPHA_0 * total_counts / total_counts.sum()

def log_odds_z(category):
    in_cat = (all_spans['category'] == category).to_numpy()
    y_i = np.asarray(span_term_matrix[in_cat].sum(axis=0)).ravel()
    y_j = total_counts - y_i
    n_i, n_j = y_i.sum(), y_j.sum()
    delta = (np.log((y_i + alpha) / (n_i + ALPHA_0 - y_i - alpha))
             - np.log((y_j + alpha) / (n_j + ALPHA_0 - y_j - alpha)))
    variance = 1 / (y_i + alpha) + 1 / (y_j + alpha)
    return pd.Series(delta / np.sqrt(variance), index=vocab)

term_z = pd.DataFrame({category: log_odds_z(category) for category in TENTATIVE_CATEGORIES})

# top characteristic terms per category (z > 1.96 is roughly significant at the 5% level)
keywords_df = pd.DataFrame({category: term_z[category].nlargest(15).index for category in TENTATIVE_CATEGORIES})

keywords_df
# export for convenience
# keywords_df.to_csv('keywords.csv', index=False)

,Parties,Document Name,Agreement Date,Expiration Date,Effective Date,License Grant,Non-Transferable License,Exclusivity,Non-Compete,Cap On Liability,Uncapped Liability
0,inc,agreement,day,date,day,grants,transferable,exclusive,competitive,damages,damages
1,collectively,sponsorship,march,effective,effective,license,non,territory,directly,liable,consequential
2,referred,maintenance,june,terminated,date,use,grants,exclusively,indirectly,consequential,liable
3,corporation,strategic,january,unless,january,hereby,license,field,competing,liability,incidental
4,company,alliance,december,term,april,exclusive,exclusive,products,engage,special,indemnification
5,llc,amendment,april,continue,june,non,solely,distribute,competitor,incidental,breach
6,parties,outsourcing,february,earlier,march,free,use,market,not,indirect,special
7,individually,development,november,commence,december,royalty,sublicense,accepts,compete,arising,punitive
8,com,branding,october,years,february,solely,hereby,subject,competitors,profits,negligence
9,ltd,endorsement,september,initial,september,worldwide,free,sell,competition,loss,liability


In [128]:
# distinctiveness score per span: mean z-score over the span's unique in-vocabulary terms
# (a mean rather than a sum, so longer spans aren't favored just for containing more terms)
candidate_term_matrix = vectorizer.transform(candidate_spans['answer_text'])

def mean_term_z(row_idx, category):
    term_idx = candidate_term_matrix[row_idx].indices
    return term_z[category].to_numpy()[term_idx].mean() if len(term_idx) else np.nan

candidate_spans['distinctiveness'] = [
    mean_term_z(i, category) for i, category in enumerate(candidate_spans['category'])
]

candidate_spans.groupby('category')['distinctiveness'].describe().reindex(TENTATIVE_CATEGORIES).round(2)

,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
Parties,1193.0,9.04,6.91,-2.85,3.59,6.87,13.57,30.53
Document Name,277.0,23.12,6.56,0.22,19.36,24.88,27.08,45.25
Agreement Date,236.0,18.18,4.19,3.95,16.78,18.83,19.61,25.08
Expiration Date,144.0,8.87,2.73,1.11,7.08,9.26,10.92,14.09
Effective Date,12.0,9.59,2.89,5.19,6.65,10.01,11.37,13.71
License Grant,234.0,4.85,2.16,-0.77,3.50,4.91,6.38,9.64
Non-Transferable License,87.0,3.45,1.54,-0.19,2.68,3.80,4.54,5.99
Exclusivity,113.0,2.01,1.05,-0.07,1.24,1.96,2.76,4.49
Non-Compete,62.0,1.87,0.69,0.15,1.39,1.82,2.30,3.82


### TF-IDF cosine similarity

In [129]:
from sklearn.feature_extraction.text import TfidfVectorizer

# IDF is fit on every answered span in the train split, so boilerplate shared across all clause types
# ("shall", "party", "agreement") is down-weighted. TF-IDF rows are L2-normalized, so a dot product is cosine similarity.
tfidf_vectorizer = TfidfVectorizer(lowercase=True, stop_words='english', sublinear_tf=True, min_df=2)
tfidf_vectorizer.fit(all_spans['answer_text'])

# context groups (from the positive context group section) so duplicate contracts count as one source
annotations['context_group_id'] = annotations['title'].map(context_group_map)

# centrality per span: mean cosine similarity to spans in other context groups of the same category (all spans,
# not just the candidates). Excluding the span's own context group stops a contract with several near-identical clauses,
# or a duplicated contract, from inflating its own spans' scores.
centrality = pd.Series(np.nan, index=annotations.index)
for category, cat_spans in annotations.groupby('category'):
    cat_tfidf = tfidf_vectorizer.transform(cat_spans['answer_text'])
    pairwise_sims = (cat_tfidf @ cat_tfidf.T).toarray()
    groups = cat_spans['context_group_id'].to_numpy()
    other_group = groups[:, None] != groups[None, :]
    centrality[cat_spans.index] = (pairwise_sims * other_group).sum(axis=1) / other_group.sum(axis=1)

# candidate_spans keeps the annotations index, so scores line up directly
candidate_spans['context_group_id'] = annotations.loc[candidate_spans.index, 'context_group_id']
candidate_spans['centrality'] = centrality[candidate_spans.index]

candidate_spans.groupby('category')['centrality'].describe().reindex(TENTATIVE_CATEGORIES).round(3)

,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
Parties,1317.0,0.005,0.008,0.000,0.000,0.002,0.006,0.038
Document Name,277.0,0.094,0.027,0.000,0.079,0.096,0.108,0.136
Agreement Date,241.0,0.056,0.021,0.003,0.042,0.053,0.072,0.122
Expiration Date,144.0,0.176,0.063,0.010,0.129,0.183,0.227,0.283
Effective Date,12.0,0.053,0.019,0.028,0.040,0.049,0.060,0.089
License Grant,234.0,0.075,0.026,0.005,0.058,0.075,0.092,0.129
Non-Transferable License,87.0,0.089,0.030,0.019,0.074,0.087,0.111,0.150
Exclusivity,113.0,0.039,0.020,0.008,0.023,0.033,0.050,0.097
Non-Compete,62.0,0.035,0.016,0.006,0.023,0.034,0.044,0.072


### Representative spans, raw output
The five candidate spans (within the IQR, with clean boundaries) scoring highest on each metric, taking at most one span per context group so a single contract can't fill the list. The two lists may overlap.

In [130]:
# EXAMPLE_CATEGORY = 'License Grant'
display_cols = ['title', 'answer_text', 'num_words', 'centrality', 'distinctiveness']

def top_spans(spans, score, n=5, cols=display_cols):
    # best-scoring span from each context group, then the top n of those. Also drop repeated span text
    # (case/whitespace-insensitive): near-duplicate filings of one agreement can land in different context groups.
    text_key = spans['answer_text'].str.lower().str.split().str.join(' ')
    return (spans.assign(text_key=text_key)
            .sort_values(score, ascending=False)
            .drop_duplicates('context_group_id')
            .drop_duplicates('text_key')
            .head(n)[cols])



# show full span text and left-align it so the excerpts are readable
def show_spans(spans, caption):
    display(
        spans.round(3).style
        .set_caption(caption)
        .set_properties(subset=['answer_text'], **{'text-align': 'left', 'white-space': 'pre-wrap', 'min-width': '500px'})
    )

for category in TENTATIVE_CATEGORIES:
    span = candidate_spans[candidate_spans['category'] == category]
    top_by_centrality = top_spans(span, 'centrality')
    top_by_distinctiveness = top_spans(span, 'distinctiveness')
    print(category)
    show_spans(top_by_centrality, f'{category}: top 5 by TF-IDF centrality')
    show_spans(top_by_distinctiveness, f'{category}: top 5 by log-odds distinctiveness')

    print('Spans in both lists:', len(top_by_centrality.index.intersection(top_by_distinctiveness.index)))

Parties


,title,answer_text,num_words,centrality,distinctiveness
2641,AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement2,Company,1.000000,0.038000,18.014000
16012,"ATHENSBANCSHARESCORP_11_02_2009-EX-1.2-AGENCY AGREEMENT , 2009","the ""Company""",2.000000,0.038000,10.462000
13163,GOLDRESOURCECORP_12_11_2008-EX-10.1-STRATEGIC ALLIANCE AGREEMENT,the Company,2.000000,0.038000,10.462000
9979,ZogenixInc_20190509_10-Q_EX-10.2_11663313_EX-10.2_Distributor Agreement,"Nippon Shinyaku Company, Ltd.",4.000000,0.038000,15.708000
14584,"TURNKEYCAPITAL,INC_07_20_2017-EX-1.1-Strategic Alliance Agreement",Seminole Indian Company,3.000000,0.038000,18.014000


,title,answer_text,num_words,centrality,distinctiveness
21439,SUNTRONCORP_05_17_2006-EX-10.22-MAINTENANCE AGREEMENT,"SUNTRON-IOWA, INC.",2.000000,0.000000,30.528000
22077,MidwestEnergyEmissionsCorp_20080604_8-K_EX-10.2_3093976_EX-10.2_Content License Agreement,"Digicorp, Inc.",2.000000,0.000000,30.528000
19322,GluMobileInc_20070319_S-1A_EX-10.09_436630_EX-10.09_Content License Agreement1,"Sorrent, Inc.",2.000000,0.001000,30.528000
17894,"BLUEHILLSBANCORP,INC_05_20_2014-EX-1.1-AGENCY AGREEMENT","Bruyette & Woods, Inc.",4.000000,0.001000,30.528000
17949,GarrettMotionInc_20181001_8-K_EX-2.4_11364532_EX-2.4_Intellectual Property Agreement,GARRETT MOTION INC.,3.000000,0.000000,30.528000


Spans in both lists: 0
Document Name


,title,answer_text,num_words,centrality,distinctiveness
20395,PFSFUNDS_06_26_2020-EX-99.H OTH MAT CONT-SERVICES AGREEMENT,SERVICES AGREEMENT,2.000000,0.136000,26.039000
3942,NANOPHASETECHNOLOGIESCORP_11_01_2005-EX-99.1-DISTRIBUTOR AGREEMENT,DISTRIBUTOR AGREEMENT,2.000000,0.135000,26.774000
3897,PACIFICSYSTEMSCONTROLTECHNOLOGYINC_08_24_2000-EX-10.53-SPONSORSHIP AGREEMENT,SPONSORSHIP AGREEMENT,2.000000,0.132000,30.730000
2192,ConformisInc_20191101_10-Q_EX-10.6_11861402_EX-10.6_Development Agreement,DEVELOPMENT AGREEMENT,2.000000,0.131000,28.704000
5215,ReedsInc_20191113_10-Q_EX-10.4_11888303_EX-10.4_Development Agreement,RECIPE DEVELOPMENT AGREEMENT,3.000000,0.131000,28.704000


...

,title,answer_text,num_words,centrality,distinctiveness
19835,MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-OUTSOURCING AGREEMENT,"In no event will IBM be liable for any lost profits, lost savings, incidental damages, or other economic consequential damages, even if IBM has been advised of the possibility of such damages.",32.000000,0.139000,10.637000
17782,RemarkHoldingsInc_20081114_10-Q_EX-10.24_2895649_EX-10.24_Content License Agreement,"EXCEPT FOR EITHER PARTY'S VIOLATION OF THE CONFIDENTIALITY OBLIGATIONS AND FOR EITHER PARTY'S INDEMNIFICATION OBLIGATIONS, IN NO EVENT WILL EITHER PARTY BE LIABLE FOR SPECIAL, INCIDENTAL, CONSEQUENTIAL, INDIRECT OR PUNITIVE DAMAGES, OR LOST PROFITS, REGARDLESS OF WHETHER SUCH LIABILITY IS BASED ON BREACH OF CONTRACT, TORT, STRICT LIABILITY, BREACH OF WARRANTIES, FAILURE OF ESSENTIAL PURPOSE OR OTHERWISE AND EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.",67.000000,0.200000,10.569000
21576,SteelVaultCorp_20081224_10-K_EX-10.16_3074935_EX-10.16_Affiliate Agreement,"IN NO EVENT WILL EQUIDATA BE LIABLE FOR ANY INDIRECT, EXEMPLARY, PUNITIVE, SPECIAL, OR CONSEQUENTIAL DAMAGES INCLUDING WITHOUT LIMITATION LOST PROFITS OR OTHER ECONOMIC LOSS, LOST REIMBURSEMENTS, AND LOST DATA, OR FOR ANY CLAIM BY ANY THIRD PARTY.",38.000000,0.151000,10.549000
6956,ScansourceInc_20190822_10-K_EX-10.38_11793958_EX-10.38_Distributor Agreement1,"INDIRECT OR CONSEQUENTIAL DAMAGES, OR LOST REVENUE, LOST PROFITS, OR LOST OR DAMAGED DATA, WHETHER ARISING IN CONTRACT, TORT (INCLUDING NEGLIGENCE), OR OTHERWISE, EVEN IF SUCH PARTY HAS BEEN INFORMED OF THE POSSIBILITY THEREOF.",34.000000,0.137000,10.525000
8000,MACROGENICSINC_08_02_2013-EX-10-COLLABORATION AGREEMENT,"NEITHER PARTY SHALL BE LIABLE TO THE OTHER FOR ANY SPECIAL, CONSEQUENTIAL, INCIDENTAL, PUNITIVE, OR INDIRECT DAMAGES OR FOR LOST PROFITS ARISING FROM OR RELATING TO ANY BREACH OF THIS AGREEMENT, REGARDLESS OF ANY NOTICE OF THE POSSIBILITY OF SUCH DAMAGES.",41.000000,0.209000,10.506000


Spans in both lists: 1
Uncapped Liability


,title,answer_text,num_words,centrality,distinctiveness
15904,"DUOSTECHNOLOGIESGROUP,INC_04_21_2009-EX-10.1-STRATEGIC ALLIANCE AGREEMENT","EXCEPT IN CONNECTION WITH A BREACH BY EITHER PARTY OF SECTION 8, NEITHER PARTY WILL BE LIABLE FOR ANY SPECIAL, INDIRECT, CONSEQUENTIAL, EXEMPLARY, PUNITIVE OR INCIDENTAL DAMAGES ARISING OUT OF OR RELATED TO THIS AGREEMENT, HOWEVER CAUSED AND UNDER ANY THEORY OF LIABILITY (INCLUDING NEGLIGENCE), EVEN IF SUCH PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.",58.000000,0.318000,7.528000
21704,PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4091580_EX-10.2_Content License Agreement_ Marketing Agreement_ Sales-Purchase Agreement1,"EXCEPT IN THE EVENT OF A BREACH OF SECTION 8 (CONFIDENTIALITY) OR LIABILITY ARISING UNDER A PARTY'S INDEMNIFICATION OBLIGATIONS UNDER SECTION 12, IN NO EVENT SHALL EITHER PARTY BE LIABLE FOR ANY INDIRECT, SPECIAL, INCIDENTAL, PUNITIVE OR CONSEQUENTIAL DAMAGES ARISING OUT OF OR IN ANY WAY CONNECTED WITH THIS AGREEMENT OR ANY MATTER RELATED HERETO, INCLUDING WITHOUT LIMITATION, LOST BUSINESS OR LOST PROFITS, EVEN IF SUCH PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.",76.000000,0.310000,7.642000
18417,IdeanomicsInc_20160330_10-K_EX-10.26_9512211_EX-10.26_Content License Agreement,"EXCEPT FOR THE ABOVE INDEMNIFICATION OBLIGATIONS AND FOR BREACHES OF SECTION 14, NEITHER PARTY SHALL BE LIABLE TO THE OTHER FOR ANY INCIDENTAL, CONSEQUENTIAL, SPECIAL OR PUNITIVE DAMAGES ARISING OUT OF THIS AGREEMENT (INCLUDING, BUT NOT LIMITED TO, LOSS OF PROFITS), WHETHER IN AN ACTION OR ARISING OUT OF BREACH OF CONTRACT, TORT OR ANY OTHER CAUSE OF ACTION EVEN IF SUCH PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.",72.000000,0.297000,6.955000
17780,RemarkHoldingsInc_20081114_10-Q_EX-10.24_2895649_EX-10.24_Content License Agreement,"EXCEPT FOR EITHER PARTY'S VIOLATION OF THE CONFIDENTIALITY OBLIGATIONS AND FOR EITHER PARTY'S INDEMNIFICATION OBLIGATIONS, IN NO EVENT WILL EITHER PARTY BE LIABLE FOR SPECIAL, INCIDENTAL, CONSEQUENTIAL, INDIRECT OR PUNITIVE DAMAGES, OR LOST PROFITS, REGARDLESS OF WHETHER SUCH LIABILITY IS BASED ON BREACH OF CONTRACT, TORT, STRICT LIABILITY, BREACH OF WARRANTIES, FAILURE OF ESSENTIAL PURPOSE OR OTHERWISE AND EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.",67.000000,0.292000,8.006000
20058,2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement,"EXCEPT IN THE EVENT OF A BREACH OF SECTION 11, NEITHER PARTY SHALL BE LIABLE FOR SPECIAL, INCIDENTAL OR CONSEQUENTIAL DAMAGES OR LOST PROFITS (HOWEVER ARISING, INCLUDING NEGLIGENCE) ARISING OUT OF OR IN CONNECTION WITH THIS AGREEMENT, EVEN IF THE PARTIES ARE AWARE OF THE POSSIBILITY OF SUCH DAMAGES.",49.000000,0.284000,7.307000


,title,answer_text,num_words,centrality,distinctiveness
17780,RemarkHoldingsInc_20081114_10-Q_EX-10.24_2895649_EX-10.24_Content License Agreement,"EXCEPT FOR EITHER PARTY'S VIOLATION OF THE CONFIDENTIALITY OBLIGATIONS AND FOR EITHER PARTY'S INDEMNIFICATION OBLIGATIONS, IN NO EVENT WILL EITHER PARTY BE LIABLE FOR SPECIAL, INCIDENTAL, CONSEQUENTIAL, INDIRECT OR PUNITIVE DAMAGES, OR LOST PROFITS, REGARDLESS OF WHETHER SUCH LIABILITY IS BASED ON BREACH OF CONTRACT, TORT, STRICT LIABILITY, BREACH OF WARRANTIES, FAILURE OF ESSENTIAL PURPOSE OR OTHERWISE AND EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.",67.000000,0.292000,8.006000
21704,PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4091580_EX-10.2_Content License Agreement_ Marketing Agreement_ Sales-Purchase Agreement1,"EXCEPT IN THE EVENT OF A BREACH OF SECTION 8 (CONFIDENTIALITY) OR LIABILITY ARISING UNDER A PARTY'S INDEMNIFICATION OBLIGATIONS UNDER SECTION 12, IN NO EVENT SHALL EITHER PARTY BE LIABLE FOR ANY INDIRECT, SPECIAL, INCIDENTAL, PUNITIVE OR CONSEQUENTIAL DAMAGES ARISING OUT OF OR IN ANY WAY CONNECTED WITH THIS AGREEMENT OR ANY MATTER RELATED HERETO, INCLUDING WITHOUT LIMITATION, LOST BUSINESS OR LOST PROFITS, EVEN IF SUCH PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.",76.000000,0.310000,7.642000
15904,"DUOSTECHNOLOGIESGROUP,INC_04_21_2009-EX-10.1-STRATEGIC ALLIANCE AGREEMENT","EXCEPT IN CONNECTION WITH A BREACH BY EITHER PARTY OF SECTION 8, NEITHER PARTY WILL BE LIABLE FOR ANY SPECIAL, INDIRECT, CONSEQUENTIAL, EXEMPLARY, PUNITIVE OR INCIDENTAL DAMAGES ARISING OUT OF OR RELATED TO THIS AGREEMENT, HOWEVER CAUSED AND UNDER ANY THEORY OF LIABILITY (INCLUDING NEGLIGENCE), EVEN IF SUCH PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.",58.000000,0.318000,7.528000
20058,2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement,"EXCEPT IN THE EVENT OF A BREACH OF SECTION 11, NEITHER PARTY SHALL BE LIABLE FOR SPECIAL, INCIDENTAL OR CONSEQUENTIAL DAMAGES OR LOST PROFITS (HOWEVER ARISING, INCLUDING NEGLIGENCE) ARISING OUT OF OR IN CONNECTION WITH THIS AGREEMENT, EVEN IF THE PARTIES ARE AWARE OF THE POSSIBILITY OF SUCH DAMAGES.",49.000000,0.284000,7.307000
1895,ImpresseCorp_20000322_S-1A_EX-10.11_5199234_EX-10.11_Co-Branding Agreement,"EXCEPT IN CONNECTION WITH A BREACH BY EITHER PARTY OF ARTICLE 7 OR SECTION 8.1.5 [REPRESENTATIONS AND WARRANTIES] (v) AND THE INDEMNIFICATION OBLIGATIONS OF IMPRESSE UNDER SECTION 9.4(i)(d) [INDEMNIFICATION BY IMPRESSE] AND THE INDEMNIFICATION OBLIGATIONS OF VERTICALNET UNDER SECTION 9.5(i)(d) [INDEMNIFICATION BY VERTICALNET], NEITHER PARTY WILL BE LIABLE FOR ANY SPECIAL, INDIRECT, CONSEQUENTIAL, EXEMPLARY OR INCIDENTAL DAMAGES ARISING OUT OF OR RELATED TO THIS AGREEMENT, HOWEVER CAUSED AND UNDER ANY THEORY OF LIABILITY (INCLUDING NEGLIGENCE), EVEN IF SUCH PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.",88.000000,0.276000,7.144000


Spans in both lists: 4


### Manual Selection for Positive Annotations
Selecting representative spans for each of the 10 categories, including surrounding context. For full-sentence annotations, this is one leading and trailing sentence.

Manual selection was performed by inspecting the top-scoring clauses on both TF-IDF centrality and log-odds distinctiveness, and selecting on the following qualities:
- Scoring top 5 on both centrality and distinctiveness
- A clean readable span, ie lack of formatting artifacts such as extra whitespace and dashes. This is overruled if the span has the previous property
- Different word content compared to other selected spans, eg covering different types of licenses, organizations, etc. This ensures that the selected representatives are not near duplicates of each other

These will be compiled into a separate report file.

In [131]:
# TODO: revise the nums; the og indices got nuked
PARTIES_SELECTED = [4034, 4151, 1804, 3609, 2684]
DOC_NAME_SELECTED = [3844, 630, 623, 373, 2807]
LICENSE_GRANT_SELECTED = [3321, 1641, 1755, 1862, 2294]
NON_TRANSFERABLE_LICENSE_SELECTED = [4502, 3510, 1873, 4369, 3324]
EXCLUSIVITY_SELECTED = [11, 1719, 1936, 2249, 3132]
NON_COMPETE_SELECTED = [2769, 1734, 807, 2435, 1849]
CAP_ON_LIABILITY_SELECTED = [1482, 3718, 2690, 3583, 4028]
UNCAPPED_LIABILITY_SELECTED = [4151, 1617, 1819, 2021, 2223]

In [132]:
# Extract the context into its down df

#### Parties Annotations

In [133]:
candidate_spans.loc[candidate_spans.index.isin(PARTIES_SELECTED), 'answer_text']

1804    January 21st, 2002
Name: answer_text, dtype: object

### Negative Source-Text Excerpts
Negative source-text excerpts are annotated spans that *do not* belong to the category in question.

For 

#### Finding negative candidates
For each target category, the candidates are annotated spans from every other CUAD category in the train split. They go through the same filters as the positives, using the target's settings:
- **Length:** within the target category's word-count IQR
- **Clean boundaries:** the target's rule (exempt if the target is a metadata category)
- **Overlap exclusion:** spans overlapping any target annotation in the same contract are removed, since CUAD frequently labels one passage with several categories
- **One per context group**, with repeated text removed, when selecting the top spans

Each candidate is then scored against the target: mean TF-IDF cosine similarity to the target's spans (in other context groups), and mean log-odds z-score using the target's characteristic terms.

In [134]:
# Negative candidates for a target category: annotated spans from every *other* category in the train split
# (all categories, not just TENTATIVE_CATEGORIES), put through the same filters as the positives and scored against
# the target, so they look like the target's positives without being one.
all_answered = df.loc[df['has_answer'], ['title', 'category', 'answer_text', 'answer_start', 'num_words']].copy()
all_answered['answer_end'] = all_answered['answer_start'] + all_answered['answer_text'].str.len()
all_answered['context_group_id'] = all_answered['title'].map(context_group_map)

def normalize_text(text):
    return text.str.lower().str.split().str.join(' ')

def negative_candidates(target):
    target_spans = all_answered[all_answered['category'] == target]
    pool = all_answered[all_answered['category'] != target]

    # same length band as the target's positives
    q1, q3 = span_length_quartiles.loc[target, ['words_q1', 'words_q3']]
    pool = pool[pool['num_words'].between(q1, q3)]

    # same boundary rule as the target's positives
    if target not in FRAGMENT_CATEGORIES:
        pool = pool[has_clean_boundaries(pool['answer_text'])]

    # exclude spans overlapping a target annotation in the same contract: CUAD often labels one passage with several
    # categories, so an overlapping span is usually a correct positive for the target, not a negative
    pairs = pool.reset_index().merge(target_spans[['title', 'answer_start', 'answer_end']], on='title', suffixes=('', '_target'))
    overlaps = (pairs['answer_start'] < pairs['answer_end_target']) & (pairs['answer_start_target'] < pairs['answer_end'])
    pool = pool.drop(pairs.loc[overlaps, 'index'].unique())

    # also exclude text identical to any target annotation elsewhere
    pool = pool[~normalize_text(pool['answer_text']).isin(set(normalize_text(target_spans['answer_text'])))]

    # similarity to target: mean TF-IDF cosine similarity to the target's spans in other context groups
    # (same definition as positive centrality, but measured against the target category instead of the span's own)
    sims = (tfidf_vectorizer.transform(pool['answer_text']) @ tfidf_vectorizer.transform(target_spans['answer_text']).T).toarray()
    other_group = pool['context_group_id'].to_numpy()[:, None] != target_spans['context_group_id'].to_numpy()[None, :]

    # target distinctiveness: mean of the target's log-odds z-scores over the span's unique terms
    pool_terms = vectorizer.transform(pool['answer_text'])
    n_terms = np.asarray(pool_terms.sum(axis=1)).ravel()
    z_sum = pool_terms @ term_z[target].to_numpy()

    return pool.assign(
        similarity_to_target=(sims * other_group).sum(axis=1) / other_group.sum(axis=1),
        target_distinctiveness=np.where(n_terms > 0, z_sum / np.maximum(n_terms, 1), np.nan),
    )

negatives_by_target = {target: negative_candidates(target) for target in TENTATIVE_CATEGORIES}

# candidates remaining per target, and which categories they come from
pd.DataFrame({
    'n_negative_candidates': {t: len(n) for t, n in negatives_by_target.items()},
    'top_source_categories': {t: ', '.join(n['category'].value_counts().head(3).index) for t, n in negatives_by_target.items()},
})

,n_negative_candidates,top_source_categories
Parties,663,"Document Name, Agreement Date, Effective Date"
Document Name,1096,"Parties, Agreement Date, Effective Date"
Agreement Date,766,"Parties, Document Name, Effective Date"
Expiration Date,1870,"Governing Law, Audit Rights, Anti-Assignment"
Effective Date,329,"Parties, Insurance, Anti-Assignment"
License Grant,2029,"Cap On Liability, Audit Rights, Insurance"
Non-Transferable License,2095,"Cap On Liability, Audit Rights, License Grant"
Exclusivity,2262,"Audit Rights, License Grant, Cap On Liability"
Non-Compete,3054,"License Grant, Audit Rights, Cap On Liability"
Cap On Liability,2451,"License Grant, Audit Rights, Insurance"


### Negative spans, raw output
The five negative candidates scoring highest on each metric for each target category.

In [135]:
negative_cols = ['title', 'category', 'answer_text', 'num_words', 'similarity_to_target', 'target_distinctiveness']

for target, negatives in negatives_by_target.items():
    top_by_similarity = top_spans(negatives, 'similarity_to_target', cols=negative_cols)
    top_by_distinctiveness = top_spans(negatives, 'target_distinctiveness', cols=negative_cols)
    print(target)
    show_spans(top_by_similarity, f'{target}: top 5 negatives by TF-IDF similarity to target')
    show_spans(top_by_distinctiveness, f'{target}: top 5 negatives by target log-odds distinctiveness')

    print('Spans in both lists:', len(top_by_similarity.index.intersection(top_by_distinctiveness.index)))

Parties


,title,category,answer_text,num_words,similarity_to_target,target_distinctiveness
3085,"OASYSMOBILE,INC_07_05_2001-EX-10.17-OUTSOURCING AGREEMENT",Anti-Assignment,Neither party may,3.000000,0.019000,3.053000
2966,FuseMedicalInc_20190321_10-K_EX-10.43_11575454_EX-10.43_Distributor Agreement,Non-Compete,The Distributor shall not:,4.000000,0.011000,-0.386000
3942,NANOPHASETECHNOLOGIESCORP_11_01_2005-EX-99.1-DISTRIBUTOR AGREEMENT,Document Name,DISTRIBUTOR AGREEMENT,2.000000,0.011000,-0.229000
19344,GluMobileInc_20070319_S-1A_EX-10.09_436630_EX-10.09_Content License Agreement1,Non-Disparagement,Licensee will,2.000000,0.010000,1.648000
12052,"LEGACYTECHNOLOGYHOLDINGS,INC_12_09_2005-EX-10.2-DISTRIBUTOR AGREEMENT",Document Name,EXCLUSIVE DISTRIBUTOR AGREEMENT,3.000000,0.009000,-1.179000


,title,category,answer_text,num_words,similarity_to_target,target_distinctiveness
3085,"OASYSMOBILE,INC_07_05_2001-EX-10.17-OUTSOURCING AGREEMENT",Anti-Assignment,Neither party may,3.000000,0.019000,3.053000
8109,ELANDIAINTERNATIONALINC_04_25_2007-EX-10.21-Outsourcing Agreement,Document Name,Network Management Outsourcing Agreement,4.000000,0.005000,2.446000
12101,DeltathreeInc_19991102_S-1A_EX-10.19_6227850_EX-10.19_Co-Branding Agreement_ Service Agreement,Document Name,CO-BRANDING AND SERVICES AGREEMENT,4.000000,0.003000,2.398000
16305,BEYONDCOMCORP_08_03_2000-EX-10.2-CO-HOSTING AGREEMENT,Document Name,Co-Hosting Agreement,2.000000,0.000000,1.872000
8211,PhasebioPharmaceuticalsInc_20200330_10-K_EX-10.21_12086810_EX-10.21_Development Agreement,Document Name,CO-DEVELOPMENT AGREEMENT,2.000000,0.001000,1.861000


Spans in both lists: 1
Document Name


,title,category,answer_text,num_words,similarity_to_target,target_distinctiveness
13137,RandWorldwideInc_20010402_8-KA_EX-10.2_2102464_EX-10.2_Co-Branding Agreement,License Grant,obligations under this Agreement.,4.000000,0.109000,10.084000
2966,FuseMedicalInc_20190321_10-K_EX-10.43_11575454_EX-10.43_Distributor Agreement,Non-Compete,The Distributor shall not:,4.000000,0.037000,0.124000
18206,ALLIANCEBANCORPINCOFPENNSYLVANIA_10_18_2006-EX-1.2-AGENCY AGREEMENT,Parties,Alliance Mutual Holding Company,4.000000,0.023000,3.657000
21539,SteelVaultCorp_20081224_10-K_EX-10.16_3074935_EX-10.16_Affiliate Agreement,Parties,Marketing Affiliate,2.000000,0.023000,6.634000
16895,MANAKOASERVICESCORP_11_21_2007-EX-7.5-STRATEGIC ALLIANCE AGREEMENT,Parties,Manakoa Services Corporation,3.000000,0.019000,3.173000


...

,title,category,answer_text,num_words,similarity_to_target,target_distinctiveness
8202,PareteumCorp_20081001_8-K_EX-99.1_2654808_EX-99.1_Hosting Agreement,Uncapped Liability,"Limitation of liability as described in this article shall not apply: a) in case the damage or loss is caused by a Party's willful misconduct (including fraud) or gross negligence, or b) in case of a breach of a Parties obligation under article 11 (confidentiality) and article 15 (indemnification for breach of intellectual property rights).",55.000000,0.098000,5.450000
7769,"Magenta Therapeutics, Inc. - Master Development and Manufacturing Agreement",Uncapped Liability,"NOTWITHSTANDING THE FOREGOING, THESE LIMITATIONS SHALL NOT APPLY TO DAMAGES ARISING FROM A PARTY'S (I) INDEMNIFICATION OBLIGATIONS UNDER SECTION 11.1 OR SECTION 11.2 HEREOF, (II) GROSS NEGLIGENCE OR WILFUL MISCONDUCT, (III) BREACH OF ITS OBLIGATIONS UNDER SECTION 9 OR (IV) INFRINGEMENT OR MISAPPROPRIATION OF THE OTHER PARTY'S INTELLECTUAL PROPERTY.",49.000000,0.103000,5.122000
7999,MACROGENICSINC_08_02_2013-EX-10-COLLABORATION AGREEMENT,Uncapped Liability,"NOTWITHSTANDING THE FOREGOING, NOTHING IN THIS SECTION 14.7 IS INTENDED TO OR SHALL LIMIT OR RESTRICT THE INDEMNIFICATION RIGHTS OR OBLIGATIONS OF ANY PARTY UNDER ARTICLE 14, OR DAMAGES AVAILABLE FOR A PARTY'S BREACH OF CONFIDENTIALITY OBLIGATIONS IN ARTICLE 12.",40.000000,0.084000,4.627000
17087,KitovPharmaLtd_20190326_20-F_EX-4.15_11584449_EX-4.15_Manufacturing Agreement,Uncapped Liability,"Nothing in this Section ​8.7 shall operate to limit or exclude any liability under Section ​8.5 with respect to a Claim, or for fraud, or for breach by a Party of the provisions of Article ​7.",36.000000,0.079000,4.604000
11950,RevolutionMedicinesInc_20200117_S-1_EX-10.1_11948417_EX-10.1_Development Agreement,Uncapped Liability,"NOTHING IN THIS SECTION 14.5 IS INTENDED TO OR SHALL LIMIT OR RESTRICT THE INDEMNIFICATION RIGHTS OR OBLIGATIONS OF ANY PARTY UNDER SECTION 14.1 OR SECTION 14.2, OR DAMAGES AVAILABLE FOR A PARTY'S BREACH OF ITS OBLIGATIONS RELATING TO CONFIDENTIALITY UNDER ARTICLE XI OR INTELLECTUAL PROPERTY UNDER ARTICLE X.",49.000000,0.079000,4.149000


Spans in both lists: 3
Uncapped Liability


,title,category,answer_text,num_words,similarity_to_target,target_distinctiveness
13738,IpassInc_20181203_8-K_EX-99.1_11445874_EX-99.1_Reseller Agreement,Cap On Liability,"NEITHER PARTY WILL BE LIABLE FOR ANY SPECIAL, INCIDENTAL, INDIRECT, EXEMPLARY, PUNITIVE, OR CONSEQUENTIAL DAMAGES ARISING OUT OF OR IN CONNECTION WITH THIS AGREEMENT OR THE USE OF THE LICENSED SOFTWARE, THE SERVICES OR DOCUMENTATION, WHETHER FROM BREACH OF CONTRACT OR WARRANTY, FROM NEGLIGENCE, STRICT LIABILITY OR OTHER CAUSE OF ACTION, EVEN IF SUCH PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.",64.000000,0.284000,6.724000
8051,GSVINC_05_15_1998-EX-10-SPONSORSHIP AGREEMENT,Cap On Liability,"EXCEPT UNDER SECTIONS 13(a) AND 13(b), IN NO EVENT WILL EITHER PARTY BE LIABLE TO THE OTHER FOR ANY SPECIAL, INCIDENTAL OR CONSEQUENTIAL DAMAGES, WHETHER BASED ON BREACH OF CONTRACT, TORT (INCLUDING NEGLIGENCE) OR OTHERWISE, WHETHER OR NOT THAT PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGE. THE LIABILITY OF EITHER PARTY FOR DAMAGES OR ALLEGED DAMAGES HEREUNDER, WHETHER IN CONTRACT, TORT OR ANY OTHER LEGAL THEORY, IS LIMITED TO, AND WILL NOT EXCEED, THE AMOUNTS TO BE PAID BY CLIENT TO EXCITE HEREUNDER.",86.000000,0.264000,6.270000
14274,BIOFRONTERAAG_04_29_2019-EX-4.17-SUPPLY AGREEMENT,Cap On Liability,"EXCEPT AS OTHERWISE EXPRESSLY SET FORTH IN THIS AGREEMENT, IN NO EVENT WILL EITHER PARTY BE LIABLE FOR ANY SPECIAL, INDIRECT, CONSEQUENTIAL, OR INCIDENTAL DAMAGES, INCLUDING LOST PROFITS, HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, ARISING IN ANY WAY OUT OF THIS AGREEMENT.",44.000000,0.256000,6.696000
3116,"OASYSMOBILE,INC_07_05_2001-EX-10.17-OUTSOURCING AGREEMENT",Cap On Liability,"IN NO EVENT SHALL EITHER PARTY BE LIABLE FOR ANY INDIRECT, INCIDENTAL, SPECIAL OR CONSEQUENTIAL DAMAGES, OR DAMAGES FOR LOSS OF PROFITS, REVENUE, DATA OR USE, INCURRED BY THE OTHER PARTY OR ANY THIRD PARTY, WHETHER IN AN ACTION IN CONTRACT OR TORT, EVEN IF SUCH PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES. IN NO EVENT SHALL E.PIPHANY'S LIABILITY HEREUNDER EXCEED THE SUM TOTAL OF PAYMENTS MADE BY HSNS UNDER THE INITIAL TERM OF THIS AGREEMENT.",79.000000,0.250000,5.503000
6057,VERTICALNETINC_04_01_2002-EX-10.19-MAINTENANCE AND SUPPORT AGREEMENT,Cap On Liability,"TO THE MAXIMUM EXTENT PERMITTED BY LAW, IN NO EVENT SHALL EITHER PARTY BE LIABLE OR OBLIGATED IN ANY MANNER FOR ANY SPECIAL, INCIDENTAL, EXEMPLARY OR CONSEQUENTIAL DAMAGES OF ANY KIND ARISING OUT OF OR RELATING TO THIS AGREEMENT (INCLUDING, BUT NOT LIMITED TO, LOST PROFITS, REVENUES OR BUSINESS OPPORTUNITIES) HOWEVER CAUSED AND REGARDLESS OF THE FORM OF ACTION, WHETHER IN CONTRACT, TORT, NEGLIGENCE, STRICT PRODUCT LIABILITY, OR OTHERWISE, EVEN IF THE PARTY HAS BEEN INFORMED OF THE POSSIBILITY OF ANY SUCH DAMAGES IN ADVANCE.",85.000000,0.244000,5.585000


,title,category,answer_text,num_words,similarity_to_target,target_distinctiveness
13738,IpassInc_20181203_8-K_EX-99.1_11445874_EX-99.1_Reseller Agreement,Cap On Liability,"NEITHER PARTY WILL BE LIABLE FOR ANY SPECIAL, INCIDENTAL, INDIRECT, EXEMPLARY, PUNITIVE, OR CONSEQUENTIAL DAMAGES ARISING OUT OF OR IN CONNECTION WITH THIS AGREEMENT OR THE USE OF THE LICENSED SOFTWARE, THE SERVICES OR DOCUMENTATION, WHETHER FROM BREACH OF CONTRACT OR WARRANTY, FROM NEGLIGENCE, STRICT LIABILITY OR OTHER CAUSE OF ACTION, EVEN IF SUCH PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.",64.000000,0.284000,6.724000
14274,BIOFRONTERAAG_04_29_2019-EX-4.17-SUPPLY AGREEMENT,Cap On Liability,"EXCEPT AS OTHERWISE EXPRESSLY SET FORTH IN THIS AGREEMENT, IN NO EVENT WILL EITHER PARTY BE LIABLE FOR ANY SPECIAL, INDIRECT, CONSEQUENTIAL, OR INCIDENTAL DAMAGES, INCLUDING LOST PROFITS, HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, ARISING IN ANY WAY OUT OF THIS AGREEMENT.",44.000000,0.256000,6.696000
2470,VERICELCORP_08_06_2019-EX-10.10-SUPPLY AGREEMENT,Cap On Liability,"NEITHER PARTY SHALL BE LIABLE FOR ANY SPECIAL, INCIDENTAL, EXEMPLARY OR CONSEQUENTIAL DAMAGES OF ANY KIND (INCLUDING LOST PROFITS) REGARDLESS OF THE FORM OF ACTION, WHETHER IN CONTRACT, TORT, NEGLIGENCE, BREACH OF STATUTORY DUTY OR OTHERWISE, SUFFERED BY THE OTHER PARTY, EVEN IF THAT PARTY HAS BEEN INFORMED OF THE POSSIBILITY OF ANY SUCH DAMAGES IN ADVANCE. [***].",58.000000,0.230000,6.596000
16651,"VISIUMTECHNOLOGIES,INC_10_20_2004-EX-10.20-DISTRIBUTOR AGREEMENT",Cap On Liability,"Under no circumstances shall WLI be liable to Power2Ship or to any other person or entity for any incidental, special or consequential damages whether arising out of breach of warranty, breach of contract or otherwise even if WLI has been advised of the possibility of such claims or demands.",49.000000,0.163000,6.565000
19066,StampscomInc_20001114_10-Q_EX-10.47_2631630_EX-10.47_Co-Branding Agreement,Cap On Liability,"CONSEQUENTIAL, INCIDENTAL, SPECIAL, INDIRECT, OR EXEMPLARY DAMAGES ARISING OUT OF THIS AGREEMENT, EVEN IF A PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES, AND UNDER ANY CAUSE OF ACTION, INCLUDING NEGLIGENCE. THESE LIMITATIONS SHALL APPLY NOTWITHSTANDING ANY FAILURE OF ESSENTIAL PURPOSE OF ANY LIMITED REMEDY.",47.000000,0.228000,6.543000


Spans in both lists: 2
